working: 
* relationship between greenness (avg greenness OR % cover cover with NDVI > 0.7) to socioeconomic indicators (MHI) per census block group 
* compare with zoning to see where best place to put new park or green space would be 
* draw route, show elevation profile and surface / road type average per that segment / line 

In [1]:
import os
import pandas as pd
import pygris
import geo_utils as geou
import vector_utils as vu
import raster_utils as ru
import plot_utils as pu

In [2]:
county_name = "Ventura"
state_abr="CA"
county_EPSG=2229
acs_yr = 2022
acs_name="median_household_income"
out_dir="/Users/orca/Desktop/geo_files"
acs_var_file='/Users/orca/Downloads/census_dict.csv'

##############
df = pd.read_csv(acs_var_file)
acs_name_var_dict=dict(zip(df['name'], df['code']))
acs_var_id = acs_name_var_dict.get(acs_name)


In [3]:
hierarch_region="tract"
tract_var = geou.get_census_var_gdf(acs_var_id, acs_yr, hierarch_region, county_name, state_abr)
if acs_name == "total_population":
    tract_var['pop_dens'] = (tract_var[acs_var_id]/tract_var.area)/100000
    tract_var.explore(column = 'pop_dens')

## CENSUS SHAPE & VAR 
tract_var.to_file(os.path.join(out_dir, county_name.replace(" ", "")+"_"+hierarch_region+"_"+acs_name+".gpkg"))
tract_var.explore(column = acs_var_id)

Using the default year of 2021
Using FIPS code '06' for input 'CA'
Using FIPS code '111' for input 'Ventura'
Using FIPS code '06' for input 'CA'


In [ ]:
## ROADS 
county_roads = pygris.roads(state = state_abr, county = county_name, cache = True)
road_gpkg=os.path.join(out_dir, county_name.replace(" ", "")+"_roads.gpkg")
county_roads.to_file(road_gpkg)


In [4]:
## ELEVATION 

import py3dep
import rioxarray

aoi = tract_var.to_crs(county_EPSG).dissolve().geometry.iloc[0]
dem = py3dep.static_3dep_dem(geometry=aoi, crs=county_EPSG, resolution=10)
dem.rio.to_raster(os.path.join(out_dir, county_name+".tif"))
dem

<xarray.DataArray 'elevation' (y: 18214, x: 10220)> Size: 745MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)
Coordinates:
    band         int64 8B 1
  * x            (x) float64 82kB -119.6 -119.6 -119.6 ... -118.6 -118.6 -118.6
  * y            (y) float64 146kB 34.9 34.9 34.9 34.9 ... 33.21 33.21 33.21
    spatial_ref  int64 8B 0
Attributes:
    scale_factor:         1.0
    add_offset:           0.0
    _FillValue:           nan
    units:                meters
    vertical_datum:       NAVD88
    vertical_resolution:  0.001

In [ ]:
## Open Street Map

poi_name="drinking_water"
bbox=(34.4, -119.9, 35.6, -119.7)


waterftns=geou.osm_poi(bbox,poi_name)
waterftns.to_file(os.path.join(out_dir, poi_name+".gpkg"), driver="GPKG")

roads = geou.osm_roads(bbox)
roads.to_file(os.path.join(out_dir, "osm_roads.gpkg"), driver="GPKG")